# Raw Data Refactoring

DATA_SAVE_DT 기준으로 동일 시각의 TAG_SN 행들을 하나의 행(Wide 형식)으로 피벗

In [1]:
import pandas as pd

RAW_PATH = '../../data/raw/유입성상, 송풍량, 약품투입량_25.11~26.02.csv'

## 1. 원본 데이터 로드

In [2]:
df = pd.read_csv(RAW_PATH, encoding='cp949')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(20)

Shape: (1984455, 3)
Columns: ['TAG_SN', 'DATA_SAVE_DT', 'LAST_VALUE']


,TAG_SN,DATA_SAVE_DT,LAST_VALUE
0,RCS_101_RCS_AI_유량조정조A_유량,202511222352,199.912500
1,RCS_101_RCS_AI_유량조정조B_유량,202511222352,200.025000
2,RCS_201_AI_교대반응조_송풍량1,202511222352,1618.750000
3,RCS_201_AI_교대반응조_송풍량2,202511222352,1047.031250
4,RCS_201_AI_막분리조펌프A_주파수,202511222352,25.200000
5,RCS_201_AI_막분리조펌프B_주파수,202511222352,0.018750
6,RCS_201_AI_막분리조펌프C_주파수,202511222352,24.931250
7,RCS_201_AI_유량조정조_BOD,202511222352,339.599304
8,RCS_201_AI_유량조정조_COD,202511222352,174.397141
9,RCS_201_AI_유량조정조_PH,202511222352,5.437250


In [3]:
print('Unique TAG_SN:', df['TAG_SN'].nunique())
print()
for tag in df['TAG_SN'].unique():
    print(' ', tag)

Unique TAG_SN: 15

  RCS_101_RCS_AI_유량조정조A_유량
  RCS_101_RCS_AI_유량조정조B_유량
  RCS_201_AI_교대반응조_송풍량1
  RCS_201_AI_교대반응조_송풍량2
  RCS_201_AI_막분리조펌프A_주파수
  RCS_201_AI_막분리조펌프B_주파수
  RCS_201_AI_막분리조펌프C_주파수
  RCS_201_AI_유량조정조_BOD
  RCS_201_AI_유량조정조_COD
  RCS_201_AI_유량조정조_PH
  RCS_201_AI_유량조정조_SS
  RCS_201_AI_유량조정조_TN
  RCS_201_AI_유량조정조_TOC
  RCS_201_AI_유량조정조_TP
  RCS_201_AI_유량조정조_수온


## 2. 피벗 (Long → Wide)

`DATA_SAVE_DT`를 인덱스(행), `TAG_SN`을 컬럼, `LAST_VALUE`를 값으로 피벗합니다.

In [4]:
df_pivot = df.pivot_table(
    index='DATA_SAVE_DT',
    columns='TAG_SN',
    values='LAST_VALUE',
    aggfunc='mean'   # 동일 시각에 중복값이 있으면 평균 처리
)

# 컬럼 멀티인덱스 제거
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()

print('Shape:', df_pivot.shape)
df_pivot.head(10)

Shape: (132297, 16)


,DATA_SAVE_DT,RCS_101_RCS_AI_유량조정조A_유량,RCS_101_RCS_AI_유량조정조B_유량,RCS_201_AI_교대반응조_송풍량1,RCS_201_AI_교대반응조_송풍량2,RCS_201_AI_막분리조펌프A_주파수,RCS_201_AI_막분리조펌프B_주파수,RCS_201_AI_막분리조펌프C_주파수,RCS_201_AI_유량조정조_BOD,RCS_201_AI_유량조정조_COD,RCS_201_AI_유량조정조_PH,RCS_201_AI_유량조정조_SS,RCS_201_AI_유량조정조_TN,RCS_201_AI_유량조정조_TOC,RCS_201_AI_유량조정조_TP,RCS_201_AI_유량조정조_수온
0,202511222352,199.9125,200.0250,1618.75000,1047.03125,25.20000,0.01875,24.93125,339.599304,174.397141,5.437250,265.087097,81.089104,132.376297,8.02,21.365625
1,202511222353,199.3125,200.3250,1619.84375,1051.87500,25.13750,0.03750,24.94375,339.599304,174.397141,5.434625,265.087097,81.089104,132.376297,8.02,21.362500
2,202511222354,199.8000,199.6500,1624.84375,1047.03125,25.19375,0.01250,25.04375,339.599304,174.397141,5.439000,265.087097,81.089104,132.376297,8.02,21.378125
3,202511222355,199.7625,200.2125,1624.84375,1052.18750,25.28125,0.06250,24.93125,339.599304,174.389069,5.437250,265.087097,81.089104,132.376297,8.02,21.343750
4,202511222356,199.7625,200.4000,1624.84375,1055.62500,25.31250,0.05625,24.91250,339.599304,174.389069,5.431125,265.087097,81.089104,132.376297,8.02,21.356250
5,202511222357,200.5875,199.8375,1631.09375,1058.43750,25.25625,0.06250,25.05625,339.599304,174.389069,5.427625,265.087097,81.089104,132.376297,8.02,21.350000
6,202511222358,200.1375,200.0250,1625.93750,1050.46875,25.15000,0.10625,24.92500,339.599304,174.389069,5.433750,265.087097,81.089104,132.376297,8.02,21.356250
7,202511222359,201.1125,199.8750,1620.00000,1045.46875,25.20625,0.06250,25.06250,339.599304,174.389069,5.426750,265.087097,81.089104,132.376297,8.02,21.353125
8,202511230000,200.2500,200.2125,1623.75000,1053.59375,25.13750,0.05625,25.06875,339.599304,174.380722,5.433750,265.087097,81.089104,132.376297,8.02,21.365625
9,202511230001,0.0000,200.2125,1609.37500,1056.56250,25.19375,0.03750,24.98750,339.599304,174.380722,5.431125,265.087097,81.089104,132.376297,8.02,21.371875


## 3. DATA_SAVE_DT 파싱 및 정렬

In [5]:
df_pivot['DATA_SAVE_DT'] = pd.to_datetime(
    df_pivot['DATA_SAVE_DT'].astype(str),
    format='%Y%m%d%H%M'
)
df_pivot = df_pivot.sort_values('DATA_SAVE_DT').reset_index(drop=True)

print('기간:', df_pivot['DATA_SAVE_DT'].min(), '~', df_pivot['DATA_SAVE_DT'].max())
print('Shape:', df_pivot.shape)
df_pivot.head(10)

기간: 2025-11-22 23:52:00 ~ 2026-02-23 13:58:00
Shape: (132297, 16)


,DATA_SAVE_DT,RCS_101_RCS_AI_유량조정조A_유량,RCS_101_RCS_AI_유량조정조B_유량,RCS_201_AI_교대반응조_송풍량1,RCS_201_AI_교대반응조_송풍량2,RCS_201_AI_막분리조펌프A_주파수,RCS_201_AI_막분리조펌프B_주파수,RCS_201_AI_막분리조펌프C_주파수,RCS_201_AI_유량조정조_BOD,RCS_201_AI_유량조정조_COD,RCS_201_AI_유량조정조_PH,RCS_201_AI_유량조정조_SS,RCS_201_AI_유량조정조_TN,RCS_201_AI_유량조정조_TOC,RCS_201_AI_유량조정조_TP,RCS_201_AI_유량조정조_수온
0,2025-11-22 23:52:00,199.9125,200.0250,1618.75000,1047.03125,25.20000,0.01875,24.93125,339.599304,174.397141,5.437250,265.087097,81.089104,132.376297,8.02,21.365625
1,2025-11-22 23:53:00,199.3125,200.3250,1619.84375,1051.87500,25.13750,0.03750,24.94375,339.599304,174.397141,5.434625,265.087097,81.089104,132.376297,8.02,21.362500
2,2025-11-22 23:54:00,199.8000,199.6500,1624.84375,1047.03125,25.19375,0.01250,25.04375,339.599304,174.397141,5.439000,265.087097,81.089104,132.376297,8.02,21.378125
3,2025-11-22 23:55:00,199.7625,200.2125,1624.84375,1052.18750,25.28125,0.06250,24.93125,339.599304,174.389069,5.437250,265.087097,81.089104,132.376297,8.02,21.343750
4,2025-11-22 23:56:00,199.7625,200.4000,1624.84375,1055.62500,25.31250,0.05625,24.91250,339.599304,174.389069,5.431125,265.087097,81.089104,132.376297,8.02,21.356250
5,2025-11-22 23:57:00,200.5875,199.8375,1631.09375,1058.43750,25.25625,0.06250,25.05625,339.599304,174.389069,5.427625,265.087097,81.089104,132.376297,8.02,21.350000
6,2025-11-22 23:58:00,200.1375,200.0250,1625.93750,1050.46875,25.15000,0.10625,24.92500,339.599304,174.389069,5.433750,265.087097,81.089104,132.376297,8.02,21.356250
7,2025-11-22 23:59:00,201.1125,199.8750,1620.00000,1045.46875,25.20625,0.06250,25.06250,339.599304,174.389069,5.426750,265.087097,81.089104,132.376297,8.02,21.353125
8,2025-11-23 00:00:00,200.2500,200.2125,1623.75000,1053.59375,25.13750,0.05625,25.06875,339.599304,174.380722,5.433750,265.087097,81.089104,132.376297,8.02,21.365625
9,2025-11-23 00:01:00,0.0000,200.2125,1609.37500,1056.56250,25.19375,0.03750,24.98750,339.599304,174.380722,5.431125,265.087097,81.089104,132.376297,8.02,21.371875


## 4. 결과 확인

In [6]:
df_pivot.info()
df_pivot.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132297 entries, 0 to 132296
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   DATA_SAVE_DT              132297 non-null  datetime64[ns]
 1   RCS_101_RCS_AI_유량조정조A_유량  132297 non-null  float64       
 2   RCS_101_RCS_AI_유량조정조B_유량  132297 non-null  float64       
 3   RCS_201_AI_교대반응조_송풍량1     132297 non-null  float64       
 4   RCS_201_AI_교대반응조_송풍량2     132297 non-null  float64       
 5   RCS_201_AI_막분리조펌프A_주파수    132297 non-null  float64       
 6   RCS_201_AI_막분리조펌프B_주파수    132297 non-null  float64       
 7   RCS_201_AI_막분리조펌프C_주파수    132297 non-null  float64       
 8   RCS_201_AI_유량조정조_BOD      132297 non-null  float64       
 9   RCS_201_AI_유량조정조_COD      132297 non-null  float64       
 10  RCS_201_AI_유량조정조_PH       132297 non-null  float64       
 11  RCS_201_AI_유량조정조_SS       132297 non-null  float64       
 12  RC

,DATA_SAVE_DT,RCS_101_RCS_AI_유량조정조A_유량,RCS_101_RCS_AI_유량조정조B_유량,RCS_201_AI_교대반응조_송풍량1,RCS_201_AI_교대반응조_송풍량2,RCS_201_AI_막분리조펌프A_주파수,RCS_201_AI_막분리조펌프B_주파수,RCS_201_AI_막분리조펌프C_주파수,RCS_201_AI_유량조정조_BOD,RCS_201_AI_유량조정조_COD,RCS_201_AI_유량조정조_PH,RCS_201_AI_유량조정조_SS,RCS_201_AI_유량조정조_TN,RCS_201_AI_유량조정조_TOC,RCS_201_AI_유량조정조_TP,RCS_201_AI_유량조정조_수온
count,132297,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000,132297.000000
mean,2026-01-08 07:13:15.160888064,173.420816,175.765185,1232.590546,899.713162,25.833514,0.418205,28.386081,399.472829,182.905851,6.270753,548.505522,82.359444,238.927935,11.670100,18.638606
min,2025-11-22 23:52:00,0.000000,0.000000,2.031250,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.013625,0.000000,0.000000,0.000000,0.000000,10.337500
25%,2025-12-16 03:37:00,169.575000,169.725000,1261.718750,819.375000,24.275000,0.018750,24.868750,339.599304,21.300234,5.572875,265.087097,33.878326,132.376297,7.757366,17.490625
50%,2026-01-08 07:14:00,189.862500,189.900000,1296.875000,995.781250,25.556250,0.037500,26.000000,339.599304,174.381958,6.605375,265.087097,81.089104,132.376297,8.020000,18.593750
75%,2026-01-31 11:04:00,199.837500,199.875000,1322.500000,1057.968750,26.956250,0.056250,29.843750,454.470398,174.406281,6.853000,501.184967,81.089104,132.376297,9.013805,19.837500
max,2026-02-23 13:58:00,318.412500,290.175000,1747.031250,1332.812500,55.862500,90.243750,100.000000,1000.000000,1000.000000,9.490250,2000.000000,1000.000000,1000.000000,50.000000,23.006250
std,NaN,46.721730,41.580307,350.374185,316.486592,2.353246,5.776253,9.420988,246.953022,247.988725,0.813912,627.755621,142.238913,311.171635,12.971769,1.700672


In [7]:
# 결측값 확인
df_pivot.isnull().sum()

DATA_SAVE_DT                0
RCS_101_RCS_AI_유량조정조A_유량    0
RCS_101_RCS_AI_유량조정조B_유량    0
RCS_201_AI_교대반응조_송풍량1       0
RCS_201_AI_교대반응조_송풍량2       0
RCS_201_AI_막분리조펌프A_주파수      0
RCS_201_AI_막분리조펌프B_주파수      0
RCS_201_AI_막분리조펌프C_주파수      0
RCS_201_AI_유량조정조_BOD        0
RCS_201_AI_유량조정조_COD        0
RCS_201_AI_유량조정조_PH         0
RCS_201_AI_유량조정조_SS         0
RCS_201_AI_유량조정조_TN         0
RCS_201_AI_유량조정조_TOC        0
RCS_201_AI_유량조정조_TP         0
RCS_201_AI_유량조정조_수온         0
dtype: int64

## 5. 저장 (선택)

In [9]:
SAVE_PATH = '../../data/actual/FLOW_extended.csv'
df_pivot.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print('Saved:', SAVE_PATH)

Saved: ../../data/actual/FLOW_extended.csv


---

# 약품주입량 일별 → 30분 변환

medication1.csv (24.08~10), medication2.csv (24.11~25.02)  
단위: kg/day → kg/30min  
변환 공식: `daily_kg / 48`  (1일 = 48개 × 30분 간격 = 1440분, /1440 × 30 = /48)

## 1. 로드 및 날짜 파싱

In [2]:
import re
import pandas as pd

def parse_med1_date(s):
    """'24년8월1일' → Timestamp(2024-08-01)"""
    m = re.match(r'(\d+)년(\d+)월(\d+)일', str(s))
    if m:
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        return pd.Timestamp(2000 + y, mo, d)
    return pd.NaT

def parse_med2_date(s):
    """'11월25일' → Timestamp(2024-11-25), '01월03일' → Timestamp(2025-01-03)"""
    m = re.match(r'(\d+)월(\d+)일', str(s))
    if m:
        mo, d = int(m[1]), int(m[2])
        # Nov/Dec → 2024, Jan/Feb 이후 → 2025
        year = 2024 if mo >= 11 else 2025
        return pd.Timestamp(year, mo, d)
    return pd.NaT

med1 = pd.read_csv('../../data/actual/medication1.csv', encoding='utf-8-sig')
med2 = pd.read_csv('../../data/actual/medication2.csv', encoding='utf-8-sig')

med1.columns = ['date_raw', 'daily_kg']
med2.columns = ['date_raw', 'daily_kg']

med1['date'] = med1['date_raw'].apply(parse_med1_date)
med2['date'] = med2['date_raw'].apply(parse_med2_date)

print('medication1:', med1['date'].min().date(), '~', med1['date'].max().date(), f'({len(med1)}일)')
print('medication2:', med2['date'].min().date(), '~', med2['date'].max().date(), f'({len(med2)}일)')
pd.concat([med1.head(3), med2.head(3)], ignore_index=True)

medication1: 2024-08-01 ~ 2024-10-31 (88일)
medication2: 2024-11-25 ~ 2025-02-14 (80일)


,date_raw,daily_kg,date
0,24년8월1일,382.00,2024-08-01
1,24년8월2일,382.00,2024-08-02
2,24년8월3일,382.00,2024-08-03
3,11월25일,291.52,2024-11-25
4,11월26일,291.56,2024-11-26
5,11월27일,291.48,2024-11-27


## 2. 일별 → 30분 변환

각 날짜의 `daily_kg`를 48개의 30분 슬롯에 균등 배분  
`kg/30min = daily_kg / 48`

In [3]:
def daily_to_30min(df_daily):
    """
    일별 DataFrame(date, daily_kg) → 30분 간격 DataFrame(datetime, kg_per_30min)

    1일 = 48개 × 30분 간격
    kg/30min = daily_kg / 48  (= daily_kg / 1440 * 30)
    """
    records = []
    for _, row in df_daily.iterrows():
        for step in range(48):
            records.append({
                'datetime': row['date'] + pd.Timedelta(minutes=30 * step),
                'medication_kg_30min': row['daily_kg'] / 48,
            })
    return pd.DataFrame(records)

med1_30 = daily_to_30min(med1)
med2_30 = daily_to_30min(med2)

# 합치기 및 정렬
medication_30min = pd.concat([med1_30, med2_30], ignore_index=True)
medication_30min = medication_30min.sort_values('datetime').reset_index(drop=True)

print('Shape:', medication_30min.shape)
print('기간:', medication_30min['datetime'].min(), '~', medication_30min['datetime'].max())
medication_30min.head(10)

Shape: (8064, 2)
기간: 2024-08-01 00:00:00 ~ 2025-02-14 23:30:00


,datetime,medication_kg_30min
0,2024-08-01 00:00:00,7.958333
1,2024-08-01 00:30:00,7.958333
2,2024-08-01 01:00:00,7.958333
3,2024-08-01 01:30:00,7.958333
4,2024-08-01 02:00:00,7.958333
5,2024-08-01 02:30:00,7.958333
6,2024-08-01 03:00:00,7.958333
7,2024-08-01 03:30:00,7.958333
8,2024-08-01 04:00:00,7.958333
9,2024-08-01 04:30:00,7.958333


## 3. 결과 확인 및 공식 검증

In [4]:
# 검증: 30분 값 × 48 = 원래 일별 값 복원 확인
sample_day = medication_30min[medication_30min['datetime'].dt.date == pd.Timestamp('2024-08-01').date()]
daily_reconstructed = sample_day['medication_kg_30min'].sum()
original = med1.loc[med1['date'] == pd.Timestamp('2024-08-01'), 'daily_kg'].values[0]
print(f'[검증] 2024-08-01')
print(f'  원본 daily_kg      : {original:.2f} kg/day')
print(f'  30분 값 × 48 복원  : {daily_reconstructed:.2f} kg/day  ✓' if abs(daily_reconstructed - original) < 0.01 else '  ✗ 불일치')
print(f'  30분당 값          : {original/48:.4f} kg/30min')
print()

# gap 확인 (med1 끝 ~ med2 시작 사이 공백)
med1_end = med1['date'].max()
med2_start = med2['date'].min()
gap_days = (med2_start - med1_end).days - 1
print(f'medication1 마지막: {med1_end.date()}')
print(f'medication2 시작  : {med2_start.date()}')
print(f'공백 기간         : {gap_days}일 (2024-11-01 ~ 2024-11-24)')

[검증] 2024-08-01
  원본 daily_kg      : 382.00 kg/day
  30분 값 × 48 복원  : 382.00 kg/day  ✓
  30분당 값          : 7.9583 kg/30min

medication1 마지막: 2024-10-31
medication2 시작  : 2024-11-25
공백 기간         : 24일 (2024-11-01 ~ 2024-11-24)


In [5]:
# 저장 (선택)
SAVE_PATH = '../../data/actual/medication_30min.csv'
medication_30min.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print('Saved:', SAVE_PATH)

Saved: ../../data/actual/medication_30min.csv


---

# process1 + process2 병합

- process1: 2024-08-01 ~ 2024-10-31 (1분, 유입유량/송풍량)
- process2: 2024-11-25 ~ 2025-02-14 (1분, 유입유량/송풍량 + 펌프주파수)
- 공백 기간: 2024-11-01 ~ 2024-11-24 (24일)

> **타임스탬프 복원 방법**: data/actual 파일의 SYS_TIME은 월 단위 마커(`202408000000`)만 보존됨.  
> 각 월의 실제 시작일을 기준으로 1분 단위 순차 할당 (행 순서 유지). 월 내 결측 구간이 있으면 오차 발생 가능.

## 1. 로드 및 구조 확인

In [6]:
import pandas as pd

p1 = pd.read_csv('../../data/actual/process1.csv', encoding='cp949', low_memory=False)
p2 = pd.read_csv('../../data/actual/process2.csv', encoding='cp949', low_memory=False)

print('process1:', p1.shape, '| 기간:', p1['SYS_TIME'].min(), '~', p1['SYS_TIME'].max())
print('process2:', p2.shape, '| 기간:', p2['SYS_TIME'].min(), '~', p2['SYS_TIME'].max())
print()
print('process1 컬럼:', p1.columns.tolist())
print('process2 컬럼:', p2.columns.tolist())

process1: (124544, 5) | 기간: 202408010000 ~ 202410312359
process2: (114371, 8) | 기간: 202411250000 ~ 202502142359

process1 컬럼: ['SYS_TIME', 'flow_TankA', 'flow_TankB', 'wind1', 'wind2']
process2 컬럼: ['SYS_TIME', 'flow_TankA', 'flow_TankB', 'wind1', 'wind2', 'pumpA', 'pumpC', 'pumpD']


## 2. datetime 파싱 및 병합

In [7]:
# SYS_TIME(YYYYMMDDHHmm) → datetime
p1['datetime'] = pd.to_datetime(p1['SYS_TIME'].astype(str), format='%Y%m%d%H%M')
p2['datetime'] = pd.to_datetime(p2['SYS_TIME'].astype(str), format='%Y%m%d%H%M')

# process1에 없는 컬럼은 NaN으로 채워 병합
cols_order = ['datetime', 'flow_TankA', 'flow_TankB', 'wind1', 'wind2',
              'pumpA', 'pumpC', 'pumpD']

p1_clean = p1.drop(columns='SYS_TIME').assign(
    pumpA=float('nan'), pumpC=float('nan'), pumpD=float('nan')
)[cols_order]

p2_clean = p2.drop(columns='SYS_TIME')[cols_order]

process_merged = pd.concat([p1_clean, p2_clean], ignore_index=True)
process_merged = process_merged.sort_values('datetime').reset_index(drop=True)

print('병합 결과:')
print(f'  Shape  : {process_merged.shape}')
print(f'  기간    : {process_merged["datetime"].min()} ~ {process_merged["datetime"].max()}')
print(f'  p1 행   : {len(p1_clean):,}  (pumpA/C/D = NaN)')
print(f'  p2 행   : {len(p2_clean):,}  (pumpA/C/D 유효)')
print()
print('null 비율:')
print(process_merged.isnull().mean().round(3))
process_merged.head(5)

병합 결과:
  Shape  : (238915, 8)
  기간    : 2024-08-01 00:00:00 ~ 2025-02-14 23:59:00
  p1 행   : 124,544  (pumpA/C/D = NaN)
  p2 행   : 114,371  (pumpA/C/D 유효)

null 비율:
datetime      0.000
flow_TankA    0.000
flow_TankB    0.000
wind1         0.000
wind2         0.000
pumpA         0.521
pumpC         0.521
pumpD         0.521
dtype: float64


,datetime,flow_TankA,flow_TankB,wind1,wind2,pumpA,pumpC,pumpD
0,2024-08-01 00:00:00,139.987503,210.562500,1108.797485,1206.292480,NaN,NaN,NaN
1,2024-08-01 00:01:00,139.725006,209.250000,149.664993,1206.015015,NaN,NaN,NaN
2,2024-08-01 00:02:00,140.062500,210.149994,79.087502,1205.367554,NaN,NaN,NaN
3,2024-08-01 00:03:00,140.062500,209.437500,82.510002,1212.027466,NaN,NaN,NaN
4,2024-08-01 00:04:00,140.812500,209.774994,83.620003,1206.755005,NaN,NaN,NaN


## 3. 결과 확인 및 gap 확인

In [8]:
p1_end  = process_merged[process_merged['datetime'] < '2024-11-01']['datetime'].max()
p2_start = process_merged[process_merged['datetime'] >= '2024-11-01']['datetime'].min()
gap_days = (p2_start - p1_end).days

print(f'process1 마지막 : {p1_end}')
print(f'process2 시작   : {p2_start}')
print(f'공백 기간        : {gap_days}일  ({p1_end.date()} 다음날 ~ {p2_start.date()} 전날)')
print()
process_merged.describe()

process1 마지막 : 2024-10-31 23:59:00
process2 시작   : 2024-11-25 00:00:00
공백 기간        : 24일  (2024-10-31 다음날 ~ 2024-11-25 전날)



,datetime,flow_TankA,flow_TankB,wind1,wind2,pumpA,pumpC,pumpD
count,238915,238914.000000,238914.000000,238913.000000,238914.000000,114371.000000,114371.000000,114371.000000
mean,2024-11-07 08:59:49.329761792,162.310436,190.109649,357.881278,1259.336033,22.673147,33.446705,56.119852
min,2024-08-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-09-11 15:09:30,99.825000,199.650000,4.531250,1172.031250,23.162500,34.337500,57.550000
50%,2024-10-28 10:33:00,120.037498,200.287500,7.187500,1232.500000,23.250000,34.418750,57.681250
75%,2025-01-02 06:36:30,249.862500,209.887497,977.343750,1706.718750,23.300000,34.475000,57.756250
max,2025-02-14 23:59:00,405.524994,287.362488,1169.375000,2500.000000,79.343750,90.206250,113.393750
std,NaN,87.685577,39.861401,468.082490,531.148071,2.708964,5.025319,7.360247


## 4. 저장 (선택)

In [9]:
SAVE_PATH = '../../data/actual/process.csv'
process_merged.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print('Saved:', SAVE_PATH)

Saved: ../../data/actual/process.csv
